In [2]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

#모든 컬럼 다 보이게
pd.set_option("display.max_columns", None)
#모든 행도 다 보이게
pd.set_option("display.max_rows", None)


#판다스가 화면에 보여줄 때 잘리지 않게 출력 옵션을 바꾸기
#pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [ ]:
# 1. 파일 위치
metadata_file = "./data/metadata.csv"
data_folder = "./data/data1"
output_folder = "./outputs"

# 2. outputs 폴더 만들기
os.makedirs(output_folder, exist_ok=True)

# 3. metadata 불러오기
metadata = pd.read_csv(metadata_file)

# 4. 컬럼명 공백 제거
metadata.columns = metadata.columns.str.strip()

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


In [24]:
metadata.nunique()

type                      3
start_time             2494
ambient_temperature       5
battery_id               34
test_id                 616
uid                    7565
filename               7565
Capacity               2752
Re                     1956
Rct                    1956
dtype: int64

In [4]:
for i in range(1, 10):
    df1 = pd.read_csv(f'data/data1/0000{i}.csv')
    print(i, df1.columns)

1 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time'], dtype='str')
2 Index(['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance'], dtype='str')
3 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time'], dtype='str')
4 Index(['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance'], dtype='str')
5 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time'], dtype='str')
6 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time'], dtype='str')
7 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time'], dtype='str')
8 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time'],

1번 5번 세트 /3번 'Voltage_charge' / 2번 4번 세트

- 00001.csv → Discharge(방전) = 첫 번째 방전 기록
- 00002.csv → Impedance(임피던스) = 그 사이 상태 진단
- 00003.csv → Charge(충전) = 다음 충전 기록
- 00004.csv → Impedance(임피던스) = 또 상태 진단
- 00005.csv → Discharge(방전) = 다음 방전 기록

데이터는 한 줄 한 줄이 “일반 거래 데이터”가 아니라, 배터리의 충전/방전/임피던스 측정 사이클을 의미합니다.
=> 리튬이온 배터리를 여러 번 충전·방전·임피던스 측정한 실험 기록

Charge → Discharge = 1 cycle

- type	실험 유형 (charge, discharge, impedance)
- start_time	테스트 시작 시간
- ambient_temperature	실험 환경의 온도(°C)
- battery_id	배터리 식별 ID
- test_id	실험 ID
- uid	실험 샘플 ID
- filename	데이터 파일명
- Capacity	배터리 현재 용량(Ah)
- Re	전해질 저항(Ω)
- Rct	충전 전달 저항(Ω)
- Voltage_measured	배터리 측정 전압(V)
- Current_measured	배터리 측정 전류(A)
- Temperature_measured	배터리 실시간 온도(°C)
- Current_load	배터리 부하 전류(A)
- Voltage_load	배터리 부하 전압(V)

RUL 칼럼 값이 없어요!

**NASA 배터리 데이터에서 RUL 칼럼 추가하기(feature engineering)**

그렇다면 RUL이란?
RUL은 **현재 타임스텝(t)에서 배터리의 마지막 수명(EOL)까지 남은 사이클 수**

                                           RUL=EOL−t

도메인 지식이 꼭 필요해요!

✔**Step 1:** 배터리별 **초기 용량**을 확인하고**80% 기준** 설정

✔**Step 2:** 배터리별 **EOL (마지막 유효 사이클)** 찾기

✔**Step 3: RUL = EOL - 현재 Cycle** 계산

✔**Step 4:** 데이터셋에 **RUL 칼럼 추가**

In [5]:
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   type                 7565 non-null   str  
 1   start_time           7565 non-null   str  
 2   ambient_temperature  7565 non-null   int64
 3   battery_id           7565 non-null   str  
 4   test_id              7565 non-null   int64
 5   uid                  7565 non-null   int64
 6   filename             7565 non-null   str  
 7   Capacity             2794 non-null   str  
 8   Re                   1956 non-null   str  
 9   Rct                  1956 non-null   str  
dtypes: int64(3), str(7)
memory usage: 1.3 MB


In [ ]:
metadata['battery_id'].nunique()

34

In [10]:
metadata['battery_id'].value_counts()

battery_id
B0006    616
B0005    616
B0007    616
B0034    486
B0033    486
B0036    486
B0018    319
B0043    275
B0042    275
B0044    275
B0054    253
B0056    252
B0055    252
B0047    184
B0045    184
B0048    184
B0046    184
B0041    163
B0053    137
B0039    122
B0040    122
B0038    122
B0032     97
B0029     97
B0030     97
B0031     97
B0028     80
B0027     80
B0025     80
B0026     80
B0049     62
B0050     62
B0052     62
B0051     62
Name: count, dtype: int64

In [ ]:
summary = pd.DataFrame({
    "dtype": metadata.dtypes,
    "non_null": metadata.count(),
    "null": metadata.isnull().sum()
})

summary

,dtype,non_null,null
type,str,7565,0
start_time,str,7565,0
ambient_temperature,int64,7565,0
battery_id,str,7565,0
test_id,int64,7565,0
uid,int64,7565,0
filename,str,7565,0
Capacity,str,2794,4771
Re,str,1956,5609
Rct,str,1956,5609


- **Charge(충전)**: 배터리에 에너지를 저장하는 과정
- **Discharge(방전)**: 배터리에 저장된 에너지를 사용하는 과정
- **Impedance(내부 저항 측정)**: 배터리 내부 저항 특성을 측정하는 과정
- **Capacity(용량)**: 배터리가 저장하고 방전할 수 있는 전하량
- **RUL(Remaining Useful Life)**: 현재 시점에서 앞으로 남은 사용 가능 수명


**🔋 고려해야할 분석 요소**

- **배터리 성능 지표** → `Capacity`, `Voltage_measured`, `Temperature_measured` 변화 분석
- **충·방전 사이클** → `Cycle` 별 용량 감소 추이 확인
- **온도 영향 분석** → `ambient_temperature`가 배터리 성능에 미치는 영향
- **배터리 저항 증가 여부** → `Re`, `Rct` 값이 포함된 경우 추가 분석

In [19]:
metadata['Capacity'].value_counts().sort_values(ascending=False)

Capacity
[]                      25
0                       19
1.6743047446975208       1
1.5243662105099023       1
1.5080762969973425       1
1.4835577960067696       1
1.4671391666146525       1
1.448858156982267        1
1.4458534180949325       1
1.431118266178283        1
1.4192745516578533       1
1.3999974221808271       1
1.3885156686114437       1
1.3652234669528964       1
1.4060442272970963       1
1.4057541613525781       1
1.3867662881727878       1
1.3705053061420223       1
1.3497431114823601       1
1.3251533367939783       1
1.3111943869635805       1
1.3394234405932892       1
1.2849157142560723       1
1.281719159327807        1
1.2600051642942829       1
1.266070385993126        1
1.2413590294280803       1
1.2298873703485311       1
1.228088614506696        1
1.2139857189070042       1
1.2173435064564189       1
1.2073445944515917       1
1.18804049227045         1
1.186304749886542        1
1.2613935471028506       1
1.2648246158850653       1
1.2466494548982199 

In [23]:
metadata["Capacity"].apply(type).value_counts()

Capacity
<class 'float'>    4771
<class 'str'>      2794
Name: count, dtype: int64

In [ ]:
import pandas as pd

all_data = []

for i in range(len(metadata)):
    file_name = metadata.loc[i, "filename"]
    file_path = "data/data1/" + file_name
    
    df = pd.read_csv(file_path)
    
    df["filename"] = file_name
    df["type"] = metadata.loc[i, "type"]
    df["battery_id"] = metadata.loc[i, "battery_id"]
    df["start_time"] = metadata.loc[i, "start_time"]
    
    all_data.append(df)

full_data = pd.concat(all_data, ignore_index=True)

In [25]:
df = metadata.copy()

In [27]:
def clear_time(x):
    # 사용할 리스트
    nums = []

    # 임시 저장용
    current = ''

    # 반복문으로 숫자만 뽑아내기
    for ch in x:
        # 숫자형이거나 형태 다른 숫자 뽑아내기
        if ch.isdigit() or ch in ['.', '_', 'e', 'E', '+']:
            current += ch
        else:
            if current != '':
                nums.append(current)
                current = ''

    # 숫자 남아있을 수도 있으니 한번 더 리스트에 append
    if current != '':
        nums.append(current)

    # 연월일시분초 하면 최소 6자리, 그 이하는 버림
    if len(nums) < 6:
        return None

    # num 리스트 숫자형으로 변환, 연월일시분초이므로 앞에 6개만 숫자 변환
    try:
        nums = list(map(float, nums[:6]))
    except:
        return None

    # 각 숫자에 의미 부여
    year, month, day, hour, minute, second = nums

    # 이상치 제거
    if not (
        2000 < year
        and 1 <= month <= 12
        and 1 <= day <= 31
        and 0 <= hour < 24
        and 0 <= minute < 60
        and 0 <= second < 60
    ):
        return None

    # Timestamp로 date_time 변환
    return pd.Timestamp(
        int(year), int(month), int(day),
        int(hour), int(minute), int(second)
    )


df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time'])